# 🚀 Advanced TorchTPU: Large-Scale Training with TorchTitan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1sUNombBtQrvJGCqhyThnA_OOoHosHu-K?resourcekey=0-gz5TnsOdlAmXjkYihu0vUg)

Welcome to **Lab 3**, the final lab of the **TorchTPU DevLabs Training Workshop**!

In **Lab 1**, you trained a convolutional network from scratch. In **Lab 2**, you applied LoRA to Gemma on a single TPU core within 16 GB of HBM.

Now, we look at **TorchTitan on TPU** (`google-pytorch/torchtitan`): PyTorch's native, composable library for large-scale model pre-training and fine-tuning across multidimensional parallelism (DP, SimpleFSDP, TP, CP, and MoE).

> [NOTE]
> In this notebook, we run a **minimal single-device Qwen3 configuration** on a single TPU core so you can see TorchTitan's TPU execution loop in action with zero external downloads or tokens required.
> We then inspect the production recipes used to achieve **~50% MFU** on large TPU slices.

---
## Step 1: Environment Setup

Let's clone the open-source **`google-pytorch/torchtitan`** repository (which contains the `torchtitan/experiments/tpu` integration) and install the lightweight packages that aren't already pre-installed in Google Colab's TPU runtime.

In [ ]:
import os

# Clone google-pytorch/torchtitan (contains torchtitan/experiments/tpu)
if not os.path.exists("torchtitan/.git"):
    !git clone --depth 1 https://github.com/google-pytorch/torchtitan.git
%cd torchtitan

# Install TPU & TorchTitan dependencies in two steps (preserving Colab's pre-installed
# CPU torch, torch_tpu, libtpu, transformers, and datasets without ABI conflicts)
!pip install -q -r torchtitan/experiments/tpu/requirements.txt
!pip install -q "tyro>=1.0.5" spmd_types==0.2.5 "attn-gym[linear]==0.0.8" wandb "torch_remat @ git+https://github.com/meta-pytorch/remat.git@d302699b1c58f83fa2c7b03bc2593967e9530335"

In [ ]:
# =====================================================================
# TPU Subprocess Note:
# Because TorchTitan launches training via CLI subprocesses (!python3 / !torchrun),
# we run our sanity check in a short-lived subprocess too so the persistent Colab
# notebook kernel does not hold the exclusive libtpu device lock!
# =====================================================================
!python3 -c "import torch; device = torch.device('tpu'); res = torch.randn(256, 256, dtype=torch.bfloat16, device=device) @ torch.randn(256, 256, dtype=torch.bfloat16, device=device); torch.accelerator.synchronize(); print(f'PyTorch {torch.__version__} | Sanity check succeeded on {res.device} ({res.dtype})')"

---
## Step 2: Anatomy of a TorchTitan TPU Recipe (`ConfigRegistry`)

Modern TorchTitan uses strongly typed **Python dataclass recipes** (`TPUTrainerConfig` / `TpuGraphTrainer.Config`) rather than static TOML files, allowing clean inheritance and CLI flag overrides without touching model source code.

Let's inspect the built-in **`qwen3_debugmodel`** configuration from `torchtitan.experiments.tpu.qwen3.config_registry`, which uses the bundled `./tests/assets/tokenizer` and `c4_test` sample dataset.

In [ ]:
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")

from torchtitan.experiments.tpu.qwen3.config_registry import qwen3_debugmodel

cfg = qwen3_debugmodel()
print("Model Spec      :", cfg.model_spec.name, "/", cfg.model_spec.flavor)
print("Hidden Dim      :", cfg.model_spec.model.dim, "| Layers:", len(cfg.model_spec.model.layers), "| Vocab:", cfg.model_spec.model.vocab_size)
print("Tokenizer Path  :", cfg.hf_assets_path)
print("Dataset         :", cfg.dataloader.dataset)
print("Context Length  :", cfg.training.max_context_length)
print("Tokens / Step   :", cfg.training.num_tokens_per_microbatch_per_dp_rank)
print("Optimizer       :", cfg.optimizer)

---
## Step 3: Run the Single-Core Training Loop

We invoke TorchTitan's single-device TPU entry point (`torchtitan.experiments.tpu.train_minimal`) with `--module=torchtitan.experiments.tpu.qwen3 --config=qwen3_debugmodel` and override the microbatch token budget and step count via CLI flags.

This validates end-to-end on TPU:
1. Qwen3 `RMSNorm`, RoPE, and grouped-query `SDPA` attention execution in `bfloat16`.
2. Forward and backward autograd passes (notice Step 1 compiles the graph and Steps 2…20 run at cached wire speed!).
3. `AdamW` `foreach` optimizer updates on TPU.

In [ ]:
# Run 20-step single-core verification training on TPU
!python3 -m torchtitan.experiments.tpu.train_minimal \
    --module=torchtitan.experiments.tpu.qwen3 \
    --config=qwen3_debugmodel \
    --training.max_context_length=512 \
    --training.num_tokens_per_microbatch_per_dp_rank=2048 \
    --training.steps=20

---
## Step 4: Scaling Beyond a Single Chip — What Production Recipes Do

To scale models like **Llama 3 8B/70B** or **Qwen3 235B MoE**, multi-device parallelism is required.

Below is what the production TorchTPU integration enables on multi-chip TPU slices (v6e Trillium / Ironwood):

### 1. SimpleFSDP over DTensor
Instead of eager hook-based parameter sharding that can cause compiler graph breaks, TorchTitan's `SimpleFSDP` shards parameters, gradients, and optimizer states directly via PyTorch `DTensor` primitives across a topology-aware TPU `DeviceMesh`:
```python
# torchtitan/experiments/tpu/qwen3/infra/parallelize.py
model = apply_simple_fsdp(
    model,
    dp_mesh=parallel_dims.get_mesh("dp"),
    param_dtype=torch.bfloat16,
    reduce_dtype=torch.float32,
)
```

### 2. Whole-Step Compilation (`TpuGraphTrainer`)
With `TpuGraphTrainer` and `--compile.optimizer_compile=in_graph`, TorchTitan captures the forward pass, backward pass, and optimizer update into a single unified XLA executable—eliminating host Python dispatch overhead between layers and allowing XLA to overlap collectives with compute.

### 3. SparseCore Collective Offloading
On multi-chip TPUs equipped with **SparseCores** (`v5p`, Trillium `v6e`, and Ironwood `v7x`; note single-chip `v5e` does not include SparseCores), FSDP `reduce-scatter` collectives can be offloaded to SparseCores so **TensorCores** stay dedicated to dense matrix multiplications.

### 4. Verified Scaling Recipes (`torchtitan/experiments/tpu/recipes/`)
- **Qwen3-8B** on Ironwood `tpu7x-4x4x4` (128 TensorCore devices / 64 chips): **~51.0% MFU** at ~12,000 tokens/s/device (~24,000 tokens/s/chip).
- **Qwen3-8B** on Ironwood `tpu7x-2x2x2` (16 TensorCore devices / 8 chips): **~53.6% MFU**.
- **Qwen3-32B** on Ironwood `tpu7x-4x4x4` (128 TensorCore devices / 64 chips): **~46.5%–49.2% MFU**.
- **Llama 3 8B** on Trillium `v6e-256` (256 chips): **~39.0% MFU** at ~1.58M tokens/s across the slice.

---
## Step 5: Production Launch Command Examples (Single-Host & Multi-Host Slices)

When you move from a single Colab chip to a multi-chip TPU VM (e.g., `v6e-4`) or a GKE TPU slice, you launch the same codebase with `torchrun` and `torchtitan.experiments.tpu.train`:

```bash
# 1. Single-host 4-chip TPU VM (e.g., v6e-4) running Qwen3-1.7B with SimpleFSDP:
torchrun --nproc_per_node=4 -m torchtitan.experiments.tpu.train \
    --module=torchtitan.experiments.tpu.qwen3 \
    --config=qwen3_1_7b \
    --training.steps=10

# 2. Multi-host Ironwood slice (tpu7x-4x4x4: 16 hosts x 8 TensorCore devices = 128 devices) running Qwen3-8B (~51% MFU):
torchrun --nnodes=16 --nproc_per_node=8 \
    --rdzv_backend=static --rdzv_endpoint=$WORKERS_0_HOSTNAME:29501 --node_rank=$TPU_WORKER_ID \
    -m torchtitan.experiments.tpu.train \
    --module=torchtitan.experiments.tpu.qwen3 \
    --config=qwen3_8b \
    --parallelism.use_simple_fsdp \
    --parallelism.enable_tpu_topo_mesh \
    --compile.optimizer_compile=in_graph
```

---
### Workshop Conclusion:
Across the three labs, you progressed from:
1. **Lab 1**: Training and compiling a standard CNN from scratch on a single TPU core.
2. **Lab 2**: Fine-tuning a 2.6B Gemma model with LoRA in native `bfloat16` within 16 GB of HBM.
3. **Lab 3**: Running **TorchTitan on TPU** (`google-pytorch/torchtitan`) on a single chip and scaling up to pod-level `SimpleFSDP` recipes.